# Open in Colab
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH09/ch09_example_external_evaluation_pipelines_Langfuse.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

<!-- NOTEBOOK_METADATA source: "Jupyter Notebook" title: "Evaluate Langfuse Agent Traces with an External Evaluation Pipeline" description: "This notebook shows how to fetch representative production traces from Langfuse, score them externally, optionally push custom scores back, and turn critical failures into a replayable benchmark set for model comparison." category: "Evaluation" sidebarTitle: "External Evaluation Pipelines"-->

# Evaluate Langfuse Agent Traces with an External Evaluation Pipeline

This cookbook shows how to use Langfuse as the trace store, filtering layer, and optional score visualization layer for an external evaluation pipeline.

Tracing tells you what happened: latency, tool calls, retries, and outputs. It does not tell you whether a changed model still meets your application-specific quality bar. That is why we fetch representative traces, score them outside Langfuse, optionally push the results back in, and promote critical failures into a replayable benchmark set before model replacement.

The strongest practical design choice is to avoid making the notebook depend on LLM judging alone.

Instead, mix three kinds of signals:

- Hard checks for things you can verify directly, such as retry budget, schema validity, tool success, required handoff labels, or numeric tolerances.
- Structured rubric checks for dimensions like sufficiency or completeness.
- Trace-level judge checks when the path matters, not just the final answer.

That combination matches how Langfuse is typically most useful in production: store traces, filter representative examples, and visualize custom scores. The scoring logic itself should model good evaluation practice rather than focus on stylistic traits like tone or joyfulness.

<iframe
  width="100%"
  className="aspect-[3230/2160] rounded mt-10"
  src="https://www.youtube-nocookie.com/embed/rHfME8KDmIw?si=V4m8smxZ219AKmOU"
  title="YouTube video player"
  frameborder="0"
  allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share"
  referrerpolicy="strict-origin-when-cross-origin"
  allowFullScreen
></iframe>

---

By the end of this cookbook, you will be able to:

- Fetch representative production traces from Langfuse using time-window and tag filters.
- Define multi-layer evaluation criteria for agent workflows and outcomes.
- Score traces externally with hard checks, rubric checks, or both.
- Optionally push selected scores back into Langfuse.
- Promote critical failures into a replayable benchmark set.
- Compare candidate model versions on that benchmark set before redeployment.
- Optionally use relative trajectory ranking such as RULER when single-trace metrics are not enough.


**Note**: While we’re using a Jupyter notebook for this cookbook, in production you'd use your preferred orchestration tool. Just make sure to extract the code into a .py file and ensure all dependencies are available at runtime.



## (Prep-work) Load representative support-agent traces into Langfuse

In this demo, we will avoid a generic text-style example and instead use a lightweight support-agent workflow. Each synthetic trace will include enough structure to evaluate agent behavior: retry counts, handoff targets, required steps, tool outcomes, and the final answer.

In production, you would fetch real traces first. Here we create representative synthetic traces so the rest of the notebook can focus on the evaluation pipeline itself.

You can get your Langfuse API keys [here](https://cloud.langfuse.com/) and OpenAI API key [here](https://platform.openai.com/api-keys)

In [ ]:
%pip install langfuse openai deepeval

In [ ]:
# Store your secrets in a local .env file or your notebook environment.
# Do not hardcode credentials directly in the notebook.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Get keys for your project from the project settings page: https://cloud.langfuse.com
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_BASE_URL"] = os.getenv("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")

if not LANGFUSE_PUBLIC_KEY or not LANGFUSE_SECRET_KEY:
    raise ValueError(
        "Set LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY in your environment before running this notebook."
    )

os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    print("OPENAI_API_KEY not found.")
    OPENAI_API_KEY = input("Enter OPENAI_API_KEY: ").strip()
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OPENAI_API_KEY is set.")
print("Langfuse credentials are set.")

OPENAI_API_KEY is set.
Langfuse credentials are set.


Let's define a small set of representative support scenarios. Each case includes the workflow we expect, the required operational step, the correct handoff target, and a reference answer we can later use for correctness checks.

In [ ]:
from copy import deepcopy

from langfuse.openai import openai


def infer_issue_type(case):
    text = case.get("customer_request", "").lower()
    if "refund" in text or "charged" in text or "invoice" in text or "subscription" in text:
        return "billing_issue"
    if "ship" in text or "arrive" in text or "delivery" in text:
        return "shipping_issue"
    if "log in" in text or "password" in text or "access" in text:
        return "account_access_issue"
    return "general_support"


def infer_task_definition(case):
    return (
        f"Resolve this {case['workflow']} support request with the required steps, "
        f"correct handoff ({case['expected_handoff']}), and a clear final answer."
    )


def infer_expected_plan(case):
    plan = [case["required_step"]]
    if case["expected_handoff"] != "none":
        plan.append(f"handoff_to_{case['expected_handoff']}")
    plan.append("respond_to_customer")
    return plan


def infer_expected_tools(case):
    tools = [
        {
            "name": case["required_step"],
            "description": f"Core {case['workflow']} lookup step.",
            "input": {"case_id": case["case_id"], "workflow": case["workflow"]},
            "output": {},
        }
    ]
    if case["expected_handoff"] != "none":
        tools.append(
            {
                "name": "create_handoff",
                "description": "Escalation handoff tool.",
                "input": {"target": case["expected_handoff"], "reason": case["workflow"]},
                "output": {},
            }
        )
    return tools


def enrich_tool_call(case, call):
    return {
        "name": str(call.get("name", "unknown_tool")),
        "status": str(call.get("status", "unknown")),
        "input": call.get("input", {"case_id": case["case_id"], "request": case["customer_request"]}),
        "output": call.get("output", {"status": str(call.get("status", "unknown"))}),
        "description": call.get("description", f"Support tool in {case['workflow']} workflow."),
    }


base_cases = [
    {
        "case_id": "refund_double_charge",
        "workflow": "billing",
        "agent_type": "triage_agent",
        "customer_request": "I was charged twice for order 8129. Please fix it.",
        "required_step": "lookup_order",
        "expected_handoff": "billing_specialist",
        "reference_answer": "I found the duplicate charge and handed this to billing for refund review.",
        "max_retries": 2,
        "simulated_retry_count": 1,
        "simulated_handoff_target": "billing_specialist",
        "simulated_steps_completed": ["lookup_order", "verify_duplicate_charge", "handoff_to_billing_specialist", "respond_to_customer"],
        "simulated_tool_calls": [
            {"name": "lookup_order", "status": "success", "input": {"order_id": "8129"}},
            {"name": "create_handoff", "status": "success", "input": {"target": "billing_specialist"}},
        ],
        "simulated_final_answer": "I confirmed the duplicate charge for order 8129 and escalated this to billing for refund processing. You will receive an update within one business day.",
    },
    {
        "case_id": "shipping_status_no_handoff_needed",
        "workflow": "shipping",
        "agent_type": "triage_agent",
        "customer_request": "Where is my order 9912 and when should it arrive?",
        "required_step": "lookup_shipment",
        "expected_handoff": "none",
        "reference_answer": "Order 9912 is in transit and should arrive tomorrow.",
        "max_retries": 1,
        "simulated_retry_count": 0,
        "simulated_handoff_target": "none",
        "simulated_steps_completed": ["lookup_shipment", "respond_to_customer"],
        "simulated_tool_calls": [
            {"name": "lookup_shipment", "status": "success", "input": {"order_id": "9912"}},
        ],
        "simulated_final_answer": "I checked order 9912 and it is in transit. Carrier ETA is tomorrow.",
    },
    {
        "case_id": "reset_password_wrong_handoff",
        "workflow": "account_access",
        "agent_type": "triage_agent",
        "customer_request": "I cannot log in after changing my password.",
        "required_step": "lookup_account_status",
        "expected_handoff": "account_security",
        "reference_answer": "I checked your account and routed this to account security to restore access.",
        "max_retries": 1,
        "simulated_retry_count": 0,
        "simulated_handoff_target": "billing_specialist",
        "simulated_steps_completed": ["lookup_account_status", "handoff_to_billing_specialist", "respond_to_customer"],
        "simulated_tool_calls": [
            {"name": "lookup_account_status", "status": "success", "input": {"user_id": "user_42"}},
            {"name": "create_handoff", "status": "success", "input": {"target": "billing_specialist"}},
        ],
        "simulated_final_answer": "I escalated this and someone will follow up.",
    },
    {
        "case_id": "cancel_subscription_missing_step",
        "workflow": "billing",
        "agent_type": "triage_agent",
        "customer_request": "Please cancel my subscription at the end of this billing cycle.",
        "required_step": "lookup_subscription",
        "expected_handoff": "none",
        "reference_answer": "I located your subscription and set it to cancel at cycle end.",
        "max_retries": 1,
        "simulated_retry_count": 0,
        "simulated_handoff_target": "none",
        "simulated_steps_completed": ["respond_to_customer"],
        "simulated_tool_calls": [
            {"name": "lookup_subscription", "status": "failed", "input": {"subscription_id": "sub_01"}},
        ],
        "simulated_final_answer": "Your subscription is all set and should stop soon.",
    },
    {
        "case_id": "invoice_request_retry_exceeded",
        "workflow": "billing",
        "agent_type": "billing_agent",
        "customer_request": "Can you send me the invoice for March?",
        "required_step": "lookup_invoice",
        "expected_handoff": "none",
        "reference_answer": "I found your March invoice and sent it to your email.",
        "max_retries": 1,
        "simulated_retry_count": 3,
        "simulated_handoff_target": "none",
        "simulated_steps_completed": ["lookup_invoice", "lookup_invoice", "lookup_invoice", "respond_to_customer"],
        "simulated_tool_calls": [
            {"name": "lookup_invoice", "status": "failed", "input": {"month": "March"}},
            {"name": "lookup_invoice", "status": "failed", "input": {"month": "March"}},
            {"name": "lookup_invoice", "status": "success", "input": {"month": "March"}},
        ],
        "simulated_final_answer": "I finally found the invoice after several retries and sent it.",
    },
]

# Additional comparable scenarios so RULER can rank 3-5 similar answers per group.
comparative_variants = []
for i, tone in enumerate([
    "empathetic_and_precise",
    "neutral_and_correct",
    "vague_and_weak",
], start=1):
    template = deepcopy(base_cases[0])
    template["case_id"] = f"refund_double_charge_variant_{i}"
    if tone == "empathetic_and_precise":
        template["simulated_final_answer"] = "I understand how frustrating a double charge is. I confirmed the duplicate charge on order 8129 and escalated it to billing for immediate refund review."
    elif tone == "neutral_and_correct":
        template["simulated_final_answer"] = "Duplicate charge on order 8129 was confirmed and the case has been handed to billing for refund processing."
    else:
        template["simulated_final_answer"] = "We will look into it and get back to you."
    comparative_variants.append(template)

cases = base_cases + comparative_variants

for case in cases:
    case["task_definition"] = infer_task_definition(case)
    case["expected_plan"] = infer_expected_plan(case)
    case["expected_tools"] = infer_expected_tools(case)
    case["scenario_group_id"] = f"{case['workflow']}::{infer_issue_type(case)}::{case['expected_handoff']}"
    case["simulated_tool_calls"] = [enrich_tool_call(case, c) for c in case.get("simulated_tool_calls", [])]

print(f"Prepared enriched cases: {len(cases)}")
cases[0]


Prepared enriched cases: 8


{'case_id': 'refund_double_charge',
 'workflow': 'billing',
 'agent_type': 'triage_agent',
 'customer_request': 'I was charged twice for order 8129. Please fix it.',
 'required_step': 'lookup_order',
 'expected_handoff': 'billing_specialist',
 'reference_answer': 'I found the duplicate charge and handed this to billing for refund review.',
 'max_retries': 2,
 'simulated_retry_count': 1,
 'simulated_handoff_target': 'billing_specialist',
 'simulated_steps_completed': ['lookup_order',
  'verify_duplicate_charge',
  'handoff_to_billing_specialist',
  'respond_to_customer'],
 'simulated_tool_calls': [{'name': 'lookup_order',
   'status': 'success',
   'input': {'order_id': '8129'},
   'output': {'status': 'success'},
   'description': 'Support tool in billing workflow.'},
  {'name': 'create_handoff',
   'status': 'success',
   'input': {'target': 'billing_specialist'},
   'output': {'status': 'success'},
   'description': 'Support tool in billing workflow.'}],
 'simulated_final_answer': 'I

Next, we will log one synthetic trace per case to Langfuse. The point is not to build a realistic support agent in this prep section. The point is to create representative traces with the kinds of workflow signals an external evaluator can inspect later: retries, tool outcomes, handoffs, and the final answer.

In [ ]:
import json
from langfuse import observe, get_client, propagate_attributes

langfuse = get_client()

@observe()
def log_support_trace(case):
    payload = {
        "case_id": case["case_id"],
        "workflow": case["workflow"],
        "agent_type": case["agent_type"],
        "customer_request": case["customer_request"],
        "required_step": case["required_step"],
        "expected_handoff": case["expected_handoff"],
        "reference_answer": case["reference_answer"],
        "max_retries": case["max_retries"],
        "retry_count": case["simulated_retry_count"],
        "handoff_target": case["simulated_handoff_target"],
        "steps_completed": case["simulated_steps_completed"],
        "tool_calls": case["simulated_tool_calls"],
        "final_answer": case["simulated_final_answer"],
        "task_definition": case["task_definition"],
        "expected_plan": case["expected_plan"],
        "expected_tools": case["expected_tools"],
        "scenario_group_id": case["scenario_group_id"],
    }

    with propagate_attributes(
        tags=["ext_eval_pipelines", "support_agent", case["workflow"], case["agent_type"]],
        trace_name=f"Support case: {case['case_id']}"
    ):
        return payload

for case in cases:
    logged_trace = log_support_trace(case)
    print(json.dumps({"case_id": logged_trace["case_id"], "scenario_group_id": logged_trace["scenario_group_id"]}, indent=2))

langfuse.flush()
print("Traces flushed to Langfuse.")


{
  "case_id": "refund_double_charge",
  "scenario_group_id": "billing::billing_issue::billing_specialist"
}
{
  "case_id": "shipping_status_no_handoff_needed",
  "scenario_group_id": "shipping::shipping_issue::none"
}
{
  "case_id": "reset_password_wrong_handoff",
  "scenario_group_id": "account_access::account_access_issue::account_security"
}
{
  "case_id": "cancel_subscription_missing_step",
  "scenario_group_id": "billing::billing_issue::none"
}
{
  "case_id": "invoice_request_retry_exceeded",
  "scenario_group_id": "billing::billing_issue::none"
}
{
  "case_id": "refund_double_charge_variant_1",
  "scenario_group_id": "billing::billing_issue::billing_specialist"
}
{
  "case_id": "refund_double_charge_variant_2",
  "scenario_group_id": "billing::billing_issue::billing_specialist"
}
{
  "case_id": "refund_double_charge_variant_3",
  "scenario_group_id": "billing::billing_issue::billing_specialist"
}
Traces flushed to Langfuse.


Now you should see representative support-agent traces in the *Traces* section of the Langfuse UI.


## Why tracing alone is not enough

Tracing shows you operational behavior such as latency, retries, tool calls, and outputs. That is necessary, but it is still incomplete: when you swap models or adjust orchestration, you also need to know whether the system still satisfies workflow-specific quality expectations like taking the right handoff, completing the required step, and giving a sufficient and correct final answer.

## Fetch representative production traces

Fetching traces from Langfuse is straightforward. The important part is not just pulling any recent traces, but selecting representative ones. In practice, you will usually filter by a time window and then narrow the set by tags, workflow, agent type, or another slice that matters for the release decision you are about to make.

The example below keeps the Langfuse fetch logic and shows how you can layer additional filters in Python after retrieval.

In [ ]:
from langfuse import get_client
from datetime import datetime, timedelta
import json

BATCH_SIZE = 10
TOTAL_TRACES = 50
WORKFLOW_FILTERS = {"billing", "shipping", "returns", "account_access"}
AGENT_TYPE_FILTERS = {"triage_agent", "billing_agent", "returns_agent"}

langfuse = get_client()

now = datetime.now()
five_am_today = datetime(now.year, now.month, now.day, 5, 0)
five_am_yesterday = five_am_today - timedelta(days=1)
seven_days_ago = now - timedelta(days=7)

def parse_trace_payload(trace):
    if isinstance(trace.output, dict):
        return trace.output
    if isinstance(trace.output, str):
        return json.loads(trace.output)
    raise TypeError("Unsupported trace output format")

def is_support_case_trace(trace):
    trace_name = getattr(trace, "name", "") or ""
    if trace_name.startswith("Support case:"):
        return True

    try:
        payload = parse_trace_payload(trace)
    except Exception:
        return False

    required_keys = {"workflow", "agent_type", "required_step", "final_answer"}
    return required_keys.issubset(payload.keys())

def matches_eval_filters(trace):
    try:
        payload = parse_trace_payload(trace)
    except Exception:
        return False

    return (
        payload.get("workflow") in WORKFLOW_FILTERS
        and payload.get("agent_type") in AGENT_TYPE_FILTERS
    )

def fetch_support_traces(from_timestamp, to_timestamp):
    raw_batch = langfuse.api.trace.list(
        page=1,
        limit=BATCH_SIZE,
        tags="ext_eval_pipelines",
        from_timestamp=from_timestamp,
        to_timestamp=to_timestamp,
    ).data

    support_traces = [trace for trace in raw_batch if is_support_case_trace(trace)]
    filtered_traces = [trace for trace in support_traces if matches_eval_filters(trace)]
    return raw_batch, support_traces, filtered_traces

raw_batch, support_traces, traces_batch = fetch_support_traces(
    from_timestamp=five_am_yesterday,
    to_timestamp=now,
)

if not traces_batch:
    raw_batch, support_traces, traces_batch = fetch_support_traces(
        from_timestamp=seven_days_ago,
        to_timestamp=now,
    )
    print("No matching traces found in the last day, so the search was widened to the last 7 days.")

print(f"Tagged traces fetched: {len(raw_batch)}")
print(f"Support-case traces found: {len(support_traces)}")
print(f"Representative traces in first batch: {len(traces_batch)}")

if not traces_batch:
    raise ValueError(
        "No support-agent traces matched the evaluation filters. Re-run the prep cell above, then run this fetch cell again."
    )

Tagged traces fetched: 10
Support-case traces found: 10
Representative traces in first batch: 10


# Define multi-layer evaluation criteria

For this notebook, we will score each trace across five dimensions:

- `retry_budget_respected`
- `correct_handoff`
- `required_step_completed`
- `final_answer_sufficient`
- `final_answer_correct`

The first three are ideal hard checks. The last two are where structured rubric checks or trace-level judges are often useful.

Hard checks should do as much of the work as possible:

- `retry_count <= max_retries`
- required tool calls succeeded
- required handoff label is present when escalation is needed
- required workflow step appears in the trajectory

Then add rubric-based checks for sufficiency and correctness where a strict programmatic comparison is not enough.

Trace-level judging fits when the path matters, not just the final answer. This is where a trajectory judge or a system like RULER is a strong conceptual fit: compare trajectories against an explicit goal-achievement rubric when failure is only visible in the full path.

In the code below, we keep the production pattern simple: hard checks for workflow compliance, plus external judge-style scoring for sufficiency and correctness.

In [ ]:
from deepeval.metrics import (
    GEval,
    TaskCompletionMetric,
    ArgumentCorrectnessMetric,
    StepEfficiencyMetric,
)
from deepeval.test_case import LLMTestCaseParams, LLMTestCase, ToolCall

JUDGE_MODEL = "gpt-4o"
DEFAULT_THRESHOLD = 0.7


def normalize_retry_count(value, fallback):
    try:
        return int(value)
    except (TypeError, ValueError):
        return fallback


def normalize_list(value):
    if isinstance(value, list):
        return value
    if value is None:
        return []
    return [value]


def normalize_tool_calls(value):
    normalized_calls = []
    for call in normalize_list(value):
        if isinstance(call, dict):
            normalized_calls.append(
                {
                    "name": str(call.get("name", "unknown_tool")),
                    "status": str(call.get("status", "unknown")),
                    "input": call.get("input", {}),
                    "output": call.get("output", {}),
                    "description": str(call.get("description", "")),
                }
            )
        else:
            normalized_calls.append({"name": str(call), "status": "unknown", "input": {}, "output": {}, "description": ""})
    return normalized_calls


def normalize_expected_tools(value, required_step):
    if value:
        source = value
    else:
        source = [{"name": required_step, "input": {}, "output": {}, "description": ""}]
    normalized = []
    for tool in normalize_list(source):
        if isinstance(tool, dict):
            normalized.append(
                {
                    "name": str(tool.get("name", "unknown_tool")),
                    "input": tool.get("input", {}),
                    "output": tool.get("output", {}),
                    "description": str(tool.get("description", "")),
                }
            )
        else:
            normalized.append({"name": str(tool), "input": {}, "output": {}, "description": ""})
    return normalized


def normalize_payload(payload):
    normalized = dict(payload)
    normalized["retry_count"] = normalize_retry_count(normalized.get("retry_count"), normalized.get("max_retries", 0) + 1)
    normalized["steps_completed"] = [str(item) for item in normalize_list(normalized.get("steps_completed"))]
    normalized["tool_calls"] = normalize_tool_calls(normalized.get("tool_calls"))
    normalized["handoff_target"] = str(normalized.get("handoff_target", "none"))
    normalized["expected_handoff"] = str(normalized.get("expected_handoff", "none"))
    normalized["required_step"] = str(normalized.get("required_step", ""))
    normalized["final_answer"] = str(normalized.get("final_answer", ""))
    normalized["reference_answer"] = str(normalized.get("reference_answer", ""))
    normalized["customer_request"] = str(normalized.get("customer_request", ""))
    normalized["workflow"] = str(normalized.get("workflow", "unknown"))
    normalized["agent_type"] = str(normalized.get("agent_type", "unknown"))
    normalized["max_retries"] = normalize_retry_count(normalized.get("max_retries"), 0)
    normalized["task_definition"] = str(normalized.get("task_definition", normalized["customer_request"]))
    normalized["expected_plan"] = [str(step) for step in normalize_list(normalized.get("expected_plan", [normalized["required_step"]])) if str(step)]
    normalized["expected_tools"] = normalize_expected_tools(normalized.get("expected_tools"), normalized["required_step"])
    normalized["scenario_group_id"] = str(normalized.get("scenario_group_id", f"{normalized['workflow']}::{normalized['required_step']}::{normalized['expected_handoff']}"))
    return normalized


def payload_to_tool_calls(payload_tools):
    return [
        ToolCall(
            name=tool.get("name", "unknown_tool"),
            description=tool.get("description", ""),
            input=tool.get("input", {}),
            output=tool.get("output", {}),
        )
        for tool in payload_tools
    ]


def build_llm_test_case(payload):
    payload = normalize_payload(payload)
    return LLMTestCase(
        input=payload["customer_request"],
        actual_output=payload["final_answer"],
        tools_called=payload_to_tool_calls(payload["tool_calls"]),
        expected_tools=payload_to_tool_calls(payload["expected_tools"]),
    )


def retry_budget_respected(payload):
    payload = normalize_payload(payload)
    return payload["retry_count"] <= payload["max_retries"]


def correct_handoff(payload):
    payload = normalize_payload(payload)
    return payload["handoff_target"] == payload["expected_handoff"]


def required_step_completed(payload):
    payload = normalize_payload(payload)
    return payload["required_step"] in payload["steps_completed"]


def tool_success(payload):
    payload = normalize_payload(payload)
    relevant_calls = [call for call in payload["tool_calls"] if payload["required_step"] in call["name"]]
    if not relevant_calls:
        return False
    return all(call["status"] == "success" for call in relevant_calls)


def trajectory_summary(payload):
    payload = normalize_payload(payload)
    tool_lines = [f"- {call['name']}: {call['status']} | input={call['input']}" for call in payload["tool_calls"]]
    return f"""
Customer request: {payload['customer_request']}
Workflow: {payload['workflow']}
Agent type: {payload['agent_type']}
Task definition: {payload['task_definition']}
Expected plan: {payload['expected_plan']}
Retry count: {payload['retry_count']} of {payload['max_retries']}
Expected handoff: {payload['expected_handoff']}
Observed handoff: {payload['handoff_target']}
Required step: {payload['required_step']}
Completed steps: {', '.join(payload['steps_completed'])}
Tool calls:
{chr(10).join(tool_lines) if tool_lines else '- none recorded'}
Final answer: {payload['final_answer']}
Reference answer: {payload['reference_answer']}
""".strip()


def final_answer_sufficient(payload):
    payload = normalize_payload(payload)
    metric = GEval(
        name="final_answer_sufficient",
        criteria=(
            "Determine whether the final answer is sufficient for the customer's request. "
            "The answer should be actionable, context-aware, and should not hide missing work behind vague language."
        ),
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=JUDGE_MODEL,
    )
    test_case = LLMTestCase(input=trajectory_summary(payload), actual_output=payload["final_answer"])
    metric.measure(test_case)
    return {"score": metric.score, "reason": metric.reason}


def final_answer_correct(payload):
    payload = normalize_payload(payload)
    metric = GEval(
        name="final_answer_correct",
        criteria="Assess whether the final answer is correct given the workflow details and reference answer.",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
        model=JUDGE_MODEL,
    )
    test_case = LLMTestCase(
        input=trajectory_summary(payload),
        actual_output=payload["final_answer"],
        expected_output=payload["reference_answer"],
    )
    metric.measure(test_case)
    return {"score": metric.score, "reason": metric.reason}


def score_task_completion(payload):
    metric = TaskCompletionMetric(
        threshold=DEFAULT_THRESHOLD,
        model=JUDGE_MODEL,
        task=normalize_payload(payload)["task_definition"],
        include_reason=True,
        async_mode=False,
    )
    metric.measure(build_llm_test_case(payload))
    return {"score": float(metric.score), "reason": metric.reason}



def score_argument_correctness(payload):
    metric = ArgumentCorrectnessMetric(
        threshold=DEFAULT_THRESHOLD,
        model=JUDGE_MODEL,
        include_reason=True,
        async_mode=False,
    )
    metric.measure(build_llm_test_case(payload))
    return {"score": float(metric.score), "reason": metric.reason}


def score_step_efficiency(payload):
    payload = normalize_payload(payload)
    metric = StepEfficiencyMetric(
        threshold=DEFAULT_THRESHOLD,
        model=JUDGE_MODEL,
        include_reason=True,
        async_mode=False,
    )
    try:
        metric.measure(build_llm_test_case(payload))
        return {"score": float(metric.score), "reason": metric.reason}
    except UnboundLocalError as exc:
        # DeepEval StepEfficiencyMetric can fail internally (prompt not set) for some payloads.
        required = payload.get("required_step", "")
        steps = payload.get("steps_completed", [])
        max_retries = max(int(payload.get("max_retries", 0)), 0)
        retries = max(int(payload.get("retry_count", 0)), 0)
        base_plan_len = 2
        observed_len = len(steps)
        missing_required_penalty = 0.5 if required and required not in steps else 0.0
        extra_step_penalty = max(0, observed_len - base_plan_len) * 0.1
        retry_penalty = 0 if max_retries == 0 else min(retries / max_retries, 1.0) * 0.4
        score = max(0.0, min(1.0, 1.0 - missing_required_penalty - extra_step_penalty - retry_penalty))
        reason = (
            "Fallback score: StepEfficiencyMetric failed internally "
            f"({exc.__class__.__name__}). "
            f"Derived from required-step presence, extra steps ({observed_len}), retries ({retries}/{max_retries})."
        )
        return {"score": float(score), "reason": reason}



Wrapping evaluation logic in functions keeps the pipeline easy to test and version. Each function takes a trace payload and returns a boolean, categorical, or numeric score that can later be pushed back to Langfuse.

In [ ]:
example_payload = parse_trace_payload(traces_batch[0])

example_scores = {
    "retry_budget_respected": retry_budget_respected(example_payload),
    "correct_handoff": correct_handoff(example_payload),
    "required_step_completed": required_step_completed(example_payload),
    "required_tool_succeeded": tool_success(example_payload),
    "final_answer_sufficient": final_answer_sufficient(example_payload),
    "final_answer_correct": final_answer_correct(example_payload),
}

example_scores

Output()

Output()

{'retry_budget_respected': True,
 'correct_handoff': True,
 'required_step_completed': True,
 'required_tool_succeeded': True,
 'final_answer_sufficient': {'score': 0.43177048236403187,
  'reason': "The Actual Output acknowledges the issue but lacks specificity and actionable steps. It does not directly address the customer's request for a resolution, as it fails to mention the duplicate charge verification or the handoff to the billing specialist. While the handoff was correctly observed, the response is vague and does not provide immediate reassurance or clarity on the next steps, unlike the Reference answer."},
 'final_answer_correct': {'score': 0.6528943405057316,
  'reason': "The Actual Output does not match the Expected Output, as it lacks the confirmation of finding the duplicate charge and the handoff for refund review. However, the workflow was followed correctly with all required steps completed, including the lookup and handoff. The discrepancy in the final answer is not jus

# Score traces externally

Use external judge logic, hard checks, or both. The important thing is that the score logic lives outside the tracing system, so it can evolve with your application. Langfuse can store the trace and optionally store the scores, but your evaluator defines what good behavior means.

You can use any evaluation library. `deepeval` is sufficient here for independent checks, while RULER-style trajectory judging is a good next step when you need explicit whole-trajectory comparisons.

In [ ]:
evaluated_traces = []

for trace in traces_batch:
    payload = normalize_payload(parse_trace_payload(trace))

    sufficiency = final_answer_sufficient(payload)
    correctness = final_answer_correct(payload)
    task_completion = score_task_completion(payload)
    argument_correctness = score_argument_correctness(payload)
    step_efficiency = score_step_efficiency(payload)

    evaluated_traces.append(
        {
            "trace_id": trace.id,
            "case_id": payload["case_id"],
            "workflow": payload["workflow"],
            "agent_type": payload["agent_type"],
            "scenario_group_id": payload["scenario_group_id"],
            "payload": payload,
            "scores": {
                "retry_budget_respected": float(retry_budget_respected(payload)),
                "correct_handoff": float(correct_handoff(payload)),
                "required_step_completed": float(required_step_completed(payload) and tool_success(payload)),
                "final_answer_sufficient": sufficiency["score"],
                "final_answer_correct": correctness["score"],
                "task_completion": task_completion["score"],
                "argument_correctness": argument_correctness["score"],
                "step_efficiency": step_efficiency["score"],
            },
            "reasons": {
                "final_answer_sufficient": sufficiency["reason"],
                "final_answer_correct": correctness["reason"],
                "task_completion": task_completion["reason"],
                "argument_correctness": argument_correctness["reason"],
                "step_efficiency": step_efficiency["reason"],
            },
        }
    )

evaluated_traces[0]


Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

{'trace_id': '86712dc66d68c17630301a01cf54b8f4',
 'case_id': 'refund_double_charge_variant_3',
 'workflow': 'billing',
 'agent_type': 'triage_agent',
 'scenario_group_id': 'billing::billing_issue::billing_specialist',
 'payload': {'case_id': 'refund_double_charge_variant_3',
  'workflow': 'billing',
  'agent_type': 'triage_agent',
  'customer_request': 'I was charged twice for order 8129. Please fix it.',
  'required_step': 'lookup_order',
  'expected_handoff': 'billing_specialist',
  'reference_answer': 'I found the duplicate charge and handed this to billing for refund review.',
  'max_retries': 2,
  'retry_count': 1,
  'handoff_target': 'billing_specialist',
  'steps_completed': ['lookup_order',
   'verify_duplicate_charge',
   'handoff_to_billing_specialist',
   'respond_to_customer'],
  'tool_calls': [{'name': 'lookup_order',
    'status': 'success',
    'input': {'order_id': '8129'},
    'output': {'status': 'success'},
    'description': 'Support tool in billing workflow.'},
   

For LLM-based checks, keep the reasons. They are useful for debugging and for reviewing borderline cases later. For hard checks, the boolean itself is often enough, but for judge-based scores the explanation becomes part of the audit trail.

## 5. Optional: push scores back into Langfuse

This step is optional. If you want the pipeline to remain fully independent, you can stop after exporting traces, scoring them externally, and writing your benchmark set to disk or another store.

Langfuse becomes especially useful again if you want score visualization and filtering in the same system that stores the traces. In that case, attach booleans, numeric scores, and comments back to the original trace for follow-up analysis.

In [ ]:
WRITE_SCORES_TO_LANGFUSE = False

if WRITE_SCORES_TO_LANGFUSE:
    for evaluated in evaluated_traces:
        for metric_name, metric_value in evaluated["scores"].items():
            kwargs = {
                "trace_id": evaluated["trace_id"],
                "name": metric_name,
                "value": float(metric_value),
            }
            if metric_name in evaluated["reasons"]:
                kwargs["comment"] = evaluated["reasons"][metric_name]
            langfuse.create_score(**kwargs)

    print(f"Pushed scores for {len(evaluated_traces)} traces.")
else:
    print("Skipping Langfuse score write-back. Set WRITE_SCORES_TO_LANGFUSE = True to enable it.")


Skipping Langfuse score write-back. Set WRITE_SCORES_TO_LANGFUSE = True to enable it.


#  Promote critical failures into a replayable benchmark set

This is the missing bridge from tracing to regression testing. A trace becomes much more valuable once a critical failure is promoted into a replayable benchmark case with its workflow metadata, reference answer, prior scores, and evaluator notes.

That benchmark set is also where you can start tracking richer text-quality criteria. Independent metrics like `deepeval` can score sufficiency and correctness one trace at a time. Later, if you want to compare several candidate trajectories for the same benchmark case, you can add a relative judge such as RULER on top.

In [ ]:
import json

critical_failures = [
    evaluated for evaluated in evaluated_traces
    if (
        evaluated["scores"]["retry_budget_respected"] < 1
        or evaluated["scores"]["correct_handoff"] < 1
        or evaluated["scores"]["required_step_completed"] < 1
        or evaluated["scores"]["final_answer_sufficient"] < DEFAULT_THRESHOLD
        or evaluated["scores"]["final_answer_correct"] < DEFAULT_THRESHOLD
        or evaluated["scores"]["task_completion"] < DEFAULT_THRESHOLD
        or evaluated["scores"]["argument_correctness"] < DEFAULT_THRESHOLD
        or evaluated["scores"]["step_efficiency"] < DEFAULT_THRESHOLD
    )
]

benchmark_set_path = "benchmark_set.jsonl"

with open(benchmark_set_path, "w", encoding="utf-8") as benchmark_file:
    for failure in critical_failures:
        benchmark_file.write(
            json.dumps(
                {
                    "case_id": failure["case_id"],
                    "workflow": failure["workflow"],
                    "agent_type": failure["agent_type"],
                    "scenario_group_id": failure["scenario_group_id"],
                    "benchmark_type": "critical_failure_regression",
                    "payload": failure["payload"],
                    "reference_answer": failure["payload"]["reference_answer"],
                    "scores": failure["scores"],
                    "reasons": failure["reasons"],
                    "text_quality_dimensions": [
                        "final_answer_sufficient",
                        "final_answer_correct",
                        "task_completion",
                        "argument_correctness",
                    ],
                }
            ) + "\n"
        )

print(f"Benchmark cases written: {len(critical_failures)}")
print(f"Benchmark set path: {benchmark_set_path}")


Benchmark cases written: 10
Benchmark set path: benchmark_set.jsonl


# Compare candidate model versions on the benchmark set

Now the benchmark set can be used as a lightweight regression suite. Instead of promoting a new model because it looks good on a few spot checks, run the benchmark cases again, rescore them externally, and compare aggregate results before redeployment.

The code cell below replays the benchmark cases against candidate models and summarizes the resulting scores. In practice, you would run this step before changing the production model, the routing logic, or the tool orchestration.

This still scores each run independently. That is useful, but sometimes not enough. If several candidate trajectories all look plausible and you want a judge to rank them relative to one another on goal achievement or text quality, add a relative evaluator such as RULER on top of the benchmark set.

In [ ]:
import statistics

with open(benchmark_set_path, "r", encoding="utf-8") as benchmark_file:
    benchmark_cases = [json.loads(line) for line in benchmark_file]


def run_candidate_agent(payload, model_name):
    payload = normalize_payload(payload)

    prompt = f"""
You are a customer support agent. Return JSON only with these keys:
retry_count, handoff_target, steps_completed, tool_calls, final_answer.

Constraints:
- steps_completed must be a JSON array of strings.
- tool_calls must be a JSON array of objects with keys: name, status, input, output, description.
- retry_count must be an integer.
- handoff_target must be a string.
- final_answer must be a string.

Task definition: {payload['task_definition']}
Expected plan: {payload['expected_plan']}
Expected tools: {payload['expected_tools']}
Customer request: {payload['customer_request']}
Workflow: {payload['workflow']}
Required step: {payload['required_step']}
Expected handoff: {payload['expected_handoff']}
Reference answer: {payload['reference_answer']}
""".strip()

    response = openai.chat.completions.create(
        model=model_name,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
    )

    candidate = json.loads(response.choices[0].message.content)
    return normalize_payload(
        {
            **payload,
            "retry_count": candidate.get("retry_count", payload["max_retries"] + 1),
            "handoff_target": candidate.get("handoff_target", "none"),
            "steps_completed": candidate.get("steps_completed", []),
            "tool_calls": candidate.get("tool_calls", []),
            "final_answer": candidate.get("final_answer", ""),
        }
    )


def evaluate_candidate_model(model_name, benchmark_cases):
    model_scores = []
    for benchmark_case in benchmark_cases:
        payload = run_candidate_agent(benchmark_case["payload"], model_name)
        sufficiency = final_answer_sufficient(payload)
        correctness = final_answer_correct(payload)
        task_completion = score_task_completion(payload)
        argument_correctness = score_argument_correctness(payload)
        step_efficiency = score_step_efficiency(payload)

        model_scores.append(
            {
                "retry_budget_respected": float(retry_budget_respected(payload)),
                "correct_handoff": float(correct_handoff(payload)),
                "required_step_completed": float(required_step_completed(payload) and tool_success(payload)),
                "final_answer_sufficient": sufficiency["score"],
                "final_answer_correct": correctness["score"],
                "task_completion": task_completion["score"],
                "argument_correctness": argument_correctness["score"],
                "step_efficiency": step_efficiency["score"],
            }
        )

    if not model_scores:
        return {"model": model_name, "cases": 0}

    keys = model_scores[0].keys()
    return {
        "model": model_name,
        "cases": len(model_scores),
        **{key: statistics.mean(score[key] for score in model_scores) for key in keys},
    }


model_comparison = [
    evaluate_candidate_model("gpt-5.4-mini", benchmark_cases),
    evaluate_candidate_model("gpt-5.4-nano", benchmark_cases),
]

model_comparison


Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

[{'model': 'gpt-5.4-mini',
  'cases': 10,
  'retry_budget_respected': 1.0,
  'correct_handoff': 1.0,
  'required_step_completed': 0.3,
  'final_answer_sufficient': 0.9973172197306286,
  'final_answer_correct': 1.0,
  'task_completion': 0.685,
  'argument_correctness': 0.0,
  'step_efficiency': 0.93},
 {'model': 'gpt-5.4-nano',
  'cases': 10,
  'retry_budget_respected': 1.0,
  'correct_handoff': 1.0,
  'required_step_completed': 1.0,
  'final_answer_sufficient': 0.9787851153416378,
  'final_answer_correct': 0.8791574751323176,
  'task_completion': 0.785,
  'argument_correctness': 0.35,
  'step_efficiency': 0.95}]

This yields one coherent external evaluation loop: instrument traces, run unified per-trace metrics, export one benchmark set, compare candidate models, and use comparative RULER ranking to spot weak answers among similar cases.
